In [1]:
import os
import re
import json
import logging
from tqdm import tqdm
import pandas as pd

In [2]:
# =============== 配置区域 ===============
LOG_DIR = "../data/wget/raw"   # TODO: 改成你的日志目录，包含150个 .log 文件
OUTPUT_CSV = "stats.csv"           # 输出文件名
VERBOSE = True                     # True: 打印更多调试信息
# =======================================

logging.basicConfig(level=logging.DEBUG if VERBOSE else logging.WARNING,
                    format="%(levelname)s: %(message)s")

MAL_RE = re.compile(r"^wget-baseline-attack-(\d+)\.log$")
BEN_RE = re.compile(r"^wget-normal-(\d+)\.log$")

EDGE_FIELDS = ("used", "wasGeneratedBy", "wasInformedBy", "wasDerivedFrom")


def label_from_name(name: str) -> str:
    if MAL_RE.search(name):
        return "malicious"
    if BEN_RE.search(name):
        return "benign"
    return "unknown"


def parse_nodes(json_string: str, node_map: dict):
    """解析CamFlow JSON字符串中的节点("activity"或"entity")。
    解析结果写入 node_map: { uid -> prov:type }。
    行为参考你给的示例实现，但做了更稳健的异常处理（遇到坏行仅跳过）。
    """
    try:
        json_object = json.loads(json_string)
    except Exception as e:
        if VERBOSE:
            logging.debug("JSON decode error: %s\nline[:200]=%r", e, json_string[:200])
        return  # 跳过异常行

    if "activity" in json_object:
        activity = json_object["activity"]
        if isinstance(activity, dict):
            for uid, meta in activity.items():
                if uid not in node_map:
                    if isinstance(meta, dict) and ("prov:type" in meta):
                        node_map[uid] = meta["prov:type"]
                    else:
                        if VERBOSE:
                            logging.debug("skip activity without prov:type: %s", uid)

    if "entity" in json_object:
        entity = json_object["entity"]
        if isinstance(entity, dict):
            for uid, meta in entity.items():
                if uid not in node_map:
                    if isinstance(meta, dict) and ("prov:type" in meta):
                        node_map[uid] = meta["prov:type"]
                    else:
                        if VERBOSE:
                            logging.debug("skip entity without prov:type: %s", uid)


def parse_edges(json_string: str, edge_counter: dict):
    """解析CamFlow JSON字符串中的边，累加四类关系的条数。"""
    try:
        json_object = json.loads(json_string)
    except Exception as e:
        if VERBOSE:
            logging.debug("JSON decode error: %s\nline[:200]=%r", e, json_string[:200])
        return  # 跳过异常行

    for k in EDGE_FIELDS:
        v = json_object.get(k)
        if isinstance(v, dict):
            edge_counter[k] += len(v)


def parse_one_file(filepath: str) -> dict:
    """对单个 .log 文件做统计，返回包含节点数、边数及分项的字典。"""
    node_map = {}
    edge_counter = {k: 0 for k in EDGE_FIELDS}

    # 逐行解析（假设每行是一个完整JSON对象；若不是，将被跳过）
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            parse_nodes(s, node_map)
            parse_edges(s, edge_counter)

    activity_count = sum(1 for uid, t in node_map.items() if isinstance(t, str) and t == 'task')
    entity_count = len(node_map) - activity_count  # 其余视作 entity（包括 file/process_memory 等）

    used = edge_counter['used']
    wgb = edge_counter['wasGeneratedBy']
    wif = edge_counter['wasInformedBy']
    wdf = edge_counter['wasDerivedFrom']

    return {
        "activity_count": activity_count,
        "entity_count": entity_count,
        "num_nodes": activity_count + entity_count,
        "used_count": used,
        "wasGeneratedBy_count": wgb,
        "wasInformedBy_count": wif,
        "wasDerivedFrom_count": wdf,
        "num_edges": used + wgb + wif + wdf,
    }


# %%
# 收集目标文件列表（仅限题述两类命名模式）
all_files = []
for name in os.listdir(LOG_DIR):
    if name.endswith('.log') and (MAL_RE.search(name) or BEN_RE.search(name)):
        all_files.append(os.path.join(LOG_DIR, name))
all_files.sort()

print(f"发现候选日志文件：{len(all_files)} 个（期望 150）")

# %%
# 逐文件统计
records = []
for fp in tqdm(all_files, desc="Parsing files", unit="file"):
    name = os.path.basename(fp)
    label = label_from_name(name)
    stats = parse_one_file(fp)
    rec = {"file": name, "label": label, **stats}
    records.append(rec)

# 结果表
df = pd.DataFrame(records, columns=[
    "file", "label",
    "num_nodes", "num_edges",
    "activity_count", "entity_count",
    "used_count", "wasGeneratedBy_count", "wasInformedBy_count", "wasDerivedFrom_count"
])

print("\n前5行预览：")
print(df.head())

# 保存CSV
out_path = os.path.join(LOG_DIR, OUTPUT_CSV) if not os.path.isabs(OUTPUT_CSV) else OUTPUT_CSV

df.to_csv(out_path, index=False, encoding='utf-8')
print(f"\n已保存统计结果：{out_path}")

# 汇总检查
total = len(df)
mal = (df['label'] == 'malicious').sum()
ben = (df['label'] == 'benign').sum()
print(f"总文件数：{total}（恶意 {mal}，良性 {ben}），期望 (150 = 25 恶意 + 125 良性)")

# 若需：分别按标签汇总均值/中位数
print("\n按标签聚合（均值）：")
print(df.groupby('label')[['num_nodes','num_edges','activity_count','entity_count']].mean().round(2))


发现候选日志文件：150 个（期望 150）


Parsing files: 100%|██████████| 150/150 [01:55<00:00,  1.30file/s]


前5行预览：
                          file      label  num_nodes  num_edges  \
0   wget-baseline-attack-0.log  malicious      46047     184344   
1   wget-baseline-attack-1.log  malicious      34965     126775   
2  wget-baseline-attack-10.log  malicious     113650     451351   
3  wget-baseline-attack-11.log  malicious      18417      68116   
4  wget-baseline-attack-12.log  malicious      30304     107356   

   activity_count  entity_count  used_count  wasGeneratedBy_count  \
0           12765         33282       49987                 15493   
1           10781         24184       36085                 13284   
2           31748         81902      118687                 42677   
3            6351         12066       18068                  8066   
4            9146         21158       31141                 11374   

   wasInformedBy_count  wasDerivedFrom_count  
0                12597                106267  
1                10633                 66773  
2                31571           